# Predictive Autoscaling on IBM LinuxONE

This notebook is the evidence screen for the demo video. It records the IBM LinuxONE architecture, queries the live Kubernetes gateway, and plots real request and scaling telemetry.

**Gateway:** `http://148.100.79.148:8000`

## 1. Confirm the IBM LinuxONE runtime

Run this cell first. The important evidence is the `s390x` architecture, Linux kernel, and MicroK8s version.

In [ ]:
import json
import platform
import subprocess

print(json.dumps({
    "platform": platform.platform(),
    "architecture": platform.machine(),
    "python": platform.python_version(),
    "hostname": platform.node(),
}, indent=2))
print(subprocess.run(["uname", "-a"], capture_output=True, text=True).stdout.strip())
microk8s = subprocess.run("command -v microk8s && microk8s version || echo 'MicroK8s runs on the host; see the Kubernetes evidence terminal.'", shell=True, capture_output=True, text=True)
print(microk8s.stdout.strip())

## 2. Query the live LinuxONE gateway

These are real HTTP endpoints served by the LinuxONE Kubernetes gateway.

In [ ]:
import json
import urllib.request

BASE_URL = "http://148.100.79.148:8000"

def get_json(path):
    with urllib.request.urlopen(BASE_URL + path, timeout=10) as response:
        return json.load(response)

print("health:", get_json("/health"))
print("ready:", get_json("/ready"))
print(json.dumps(get_json("/api/state"), indent=2))

## 3. Capture baseline metrics

In [ ]:
state = get_json("/api/state")
metrics = get_json("/api/metrics")
print(json.dumps({
    "observed_rps": state["rps_last_window"],
    "forecast_rps": state["forecast_rps"],
    "ready_replicas": state["ready_replicas"],
    "desired_replicas": state["desired_replicas"],
    "capacity_rps": state["capacity_rps"],
    "worker_latency_ms": metrics["latency_ms"],
}, indent=2))

## 4. Run a controlled real workload ramp

This sends real `POST /hit` requests from JupyterLab to the LinuxONE gateway. It gradually increases toward 1000 RPS while sampling the controller. Change `RAMP_SECONDS` to 60 for a shorter recording.

In [ ]:
import concurrent.futures
import time

TARGET_RPS = 1000
RAMP_SECONDS = 180
CONCURRENCY = 100
samples = []
stop_at = time.perf_counter() + RAMP_SECONDS
started = time.perf_counter()

def hit():
    request = urllib.request.Request(BASE_URL + "/hit", method="POST")
    try:
        with urllib.request.urlopen(request, timeout=15) as response:
            response.read()
            return response.status
    except Exception:
        return 0

def sample_state():
    while time.perf_counter() < stop_at:
        point = get_json("/api/state")
        samples.append({"elapsed_s": time.perf_counter() - started, **point})
        print({key: point.get(key) for key in ("rps_last_window", "forecast_rps", "desired_replicas", "ready_replicas", "pending_replicas", "capacity_rps", "action")})
        time.sleep(5)

with concurrent.futures.ThreadPoolExecutor(max_workers=CONCURRENCY) as pool:
    monitor = concurrent.futures.ThreadPoolExecutor(max_workers=1)
    monitor.submit(sample_state)
    futures = []
    while time.perf_counter() < stop_at:
        elapsed = time.perf_counter() - started
        rate = max(1, TARGET_RPS * min(1, elapsed / RAMP_SECONDS))
        futures.append(pool.submit(hit))
        time.sleep(1 / rate)
    statuses = [future.result() for future in futures]
    monitor.shutdown(wait=True)
print({"requests": len(statuses), "successful": sum(status == 200 for status in statuses)})

## 5. Plot forecast, capacity, and replicas

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

frame = pd.DataFrame(samples)
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(frame.elapsed_s, frame.rps_last_window, label="Observed RPS")
axes[0].plot(frame.elapsed_s, frame.forecast_rps, label="Forecast RPS")
axes[0].plot(frame.elapsed_s, frame.capacity_rps, label="Ready capacity")
axes[0].set_ylabel("RPS")
axes[0].legend()
axes[0].grid(alpha=.25)
axes[1].step(frame.elapsed_s, frame.desired_replicas, where="post", label="Desired replicas")
axes[1].step(frame.elapsed_s, frame.ready_replicas, where="post", label="Ready replicas")
axes[1].set_xlabel("Elapsed seconds")
axes[1].set_ylabel("Pods")
axes[1].legend()
axes[1].grid(alpha=.25)
plt.suptitle("IBM LinuxONE Predictive Autoscaling")
plt.tight_layout()
plt.show()

## 6. Final latency and scaling evidence

Use this output as the closing evidence, then stop traffic from the dashboard and show scale-down.

In [ ]:
print(json.dumps({
    "final_state": get_json("/api/state"),
    "gateway_to_worker_latency": get_json("/api/metrics"),
    "note": "Client latency includes the network path; gateway latency is measured separately.",
}, indent=2))